# **INTEGRANDES**
*   **Dady Sneider Loaiza Loaiza**
*   **Martín Santiago pinzón Quiceno**



In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy import optimize
from scipy.special import gammaln  # log de la funcion gamma (C2)
import os

C1 = 1106891817 + 1141318622
C2 = 1011099583
S = C1 + C2

In [17]:
pd.set_option("display.width", 140)  # tablas mas anchas al imprimir
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

C1 = 1106891817 + 1141318622  # suma de las dos cedulas del primer bloque
C2 = 1011099583
S = C1 + C2  # semilla y base de todos los parametros

tau = 180 + 10 * (S % 9)  # umbral del SLO en ms
epsilon = 0.01 * (1 + S % 3)  # proporcion maxima tolerada de L > tau
niveles = {0: 0.90, 1: 0.95, 2: 0.99}
confianza = niveles[S % 3]  # nivel 1 - alpha
alpha = 1 - confianza
g = 1.4 + 0.1 * (S % 7)  # factor de crecimiento de la tasa
t1 = 80 + 10 * (S % 5)  # frontera pequena / mediana en KB
t2 = 400 + 25 * (S % 5)  # frontera mediana / grande en KB
pi_prod = (4 + S % 6) / 1000  # prevalencia de incidentes en produccion
k_max = 4 + (S % 5)  # falsas alarmas maximas por dia
r = S % 6  # desplazamiento para B4
N = 10 + (S % 11)  # solicitudes consecutivas para C4

rng = np.random.default_rng(S)

In [18]:
parametros = pd.DataFrame({
    "parametro": ["S", "tau (ms)", "epsilon", "1 - alpha", "g", "t1 (KB)", "t2 (KB)", "pi", "k_max", "r", "N"],
    "valor": [str(S), str(tau), str(round(epsilon, 2)), str(confianza), str(round(g, 1)), str(t1), str(t2),
              str(pi_prod), str(k_max), str(r), str(N)],  # como texto para que no salga en notacion cientifica
})
print(parametros.to_string(index=False))

parametro      valor
        S 3259310022
 tau (ms)        180
  epsilon       0.01
1 - alpha        0.9
        g        1.7
  t1 (KB)        100
  t2 (KB)        450
       pi      0.004
    k_max          6
        r          0
        N         11


# Parte A — Auditoría y preparación de los datos

## A1. Auditoría de `solicitudes.csv`

Revisamos cada columna desde la lógica del sistema: una solicitud con caché no usa GPU, la latencia no puede ser menor que red + GPU (la espera en cola es mayor o igual a cero), los intentos van de 1 a 3 y una solicitud solo puede fallar si agotó los 3 intentos. También buscamos duplicados, nulos, valores negativos y valores fuera de escala. Primero contamos cada problema sobre los datos crudos.

In [19]:
crudos = pd.read_csv("solicitudes.csv")  # datos tal como llegan
print("filas crudas:", len(crudos))
print(crudos.head())
print()
print("nulos por columna:")
print(crudos.isna().sum())

filas crudas: 40951
   id_solicitud  t_llegada_s  tam_kb  cache_hit version  red_ms  gpu_ms  latencia_ms  intentos  exito
0             1       0.0822  308.08          0       A   13.77   49.45        63.22         1      1
1             2       0.1876  827.28          1       B   19.56    0.00        19.56         1      1
2             3       0.2032  608.65          0       B   20.98   23.39        44.37         1      1
3             4       0.2169  278.76          0       A   21.91   30.20        52.10         1      1
4             5       0.2267   84.03          1       A   20.71    0.00        20.71         1      1

nulos por columna:
id_solicitud     0
t_llegada_s      0
tam_kb           0
cache_hit        0
version          0
red_ms          41
gpu_ms          40
latencia_ms      0
intentos         0
exito            0
dtype: int64


In [20]:
duplicados_exactos = crudos.duplicated().sum()  # filas identicas en todas las columnas
ids_repetidos = crudos["id_solicitud"].duplicated().sum()  # ids que aparecen mas de una vez
print("filas duplicadas exactas:", duplicados_exactos)
print("ids repetidos:", ids_repetidos)

conteo_ids = crudos["id_solicitud"].value_counts()
ids_con_copia = conteo_ids[conteo_ids > 1].index
ids_de_exactos = set(crudos[crudos.duplicated(keep=False)]["id_solicitud"])
ids_raros = [i for i in ids_con_copia if i not in ids_de_exactos]  # id repetido pero con filas distintas
print("ids repetidos que no son copia exacta:", ids_raros)
print(crudos[crudos["id_solicitud"].isin(ids_raros)])

filas duplicadas exactas: 121
ids repetidos: 122
ids repetidos que no son copia exacta: [21359]
       id_solicitud  t_llegada_s  tam_kb  cache_hit version  red_ms  gpu_ms  latencia_ms  intentos  exito
21416         21359     718.3461  193.45          0       A    12.8     NaN       136.97         1      1
21417         21359     718.3461  193.45          0       A    12.8   14.97       136.97         1      1


In [21]:
print("latencia negativa:", (crudos["latencia_ms"] < 0).sum())
print("valores de latencia negativa:", crudos.loc[crudos["latencia_ms"] < 0, "latencia_ms"].unique())
print("red negativa:", (crudos["red_ms"] < 0).sum())
print("gpu negativa:", (crudos["gpu_ms"] < 0).sum())
print("tam_kb <= 0:", (crudos["tam_kb"] <= 0).sum())
print("gpu > 0 con cache_hit = 1:", ((crudos["gpu_ms"] > 0) & (crudos["cache_hit"] == 1)).sum())
print("gpu = 0 con cache_hit = 0:", ((crudos["gpu_ms"] == 0) & (crudos["cache_hit"] == 0)).sum())
print("intentos fuera de 1..3:", (~crudos["intentos"].isin([1, 2, 3])).sum())
print("tiempos de llegada fuera de [0, 1200]:", ((crudos["t_llegada_s"] < 0) | (crudos["t_llegada_s"] > 1200)).sum())
print("llegadas desordenadas:", (crudos["t_llegada_s"].diff() < 0).sum())
print()
print("tabla intentos vs exito:")
print(pd.crosstab(crudos["intentos"], crudos["exito"]))

latencia negativa: 122
valores de latencia negativa: [-1.]
red negativa: 0
gpu negativa: 0
tam_kb <= 0: 0
gpu > 0 con cache_hit = 1: 0
gpu = 0 con cache_hit = 0: 0
intentos fuera de 1..3: 0
tiempos de llegada fuera de [0, 1200]: 0
llegadas desordenadas: 0

tabla intentos vs exito:
exito       0      1
intentos            
1           0  34209
2           0   5491
3         242   1009


In [22]:
tamanos_ordenados = np.sort(crudos["tam_kb"].values)
print("los 70 tamanos mas grandes:")
print(tamanos_ordenados[-70:].round(1))  # se ve el salto entre ~1600 y ~17000

limite_tam = 5000  # hay un hueco enorme entre ~1600 y ~17000 KB
grandes = crudos[crudos["tam_kb"] > limite_tam]
print("filas con tam_kb > 5000:", len(grandes))
print("mayor tamano normal:", crudos.loc[crudos["tam_kb"] <= limite_tam, "tam_kb"].max())
print("menor tamano anomalo:", grandes["tam_kb"].min())
print()
print("cuantiles de los anomalos divididos por 1024:")
print((grandes["tam_kb"] / 1024).quantile([0, 0.25, 0.5, 0.75, 1]).round(1))
print("cuantiles de los tamanos normales:")
print(crudos.loc[crudos["tam_kb"] <= limite_tam, "tam_kb"].quantile([0, 0.25, 0.5, 0.75, 1]).round(1))
prueba_ks = stats.ks_2samp(grandes["tam_kb"] / 1024, crudos.loc[crudos["tam_kb"] <= limite_tam, "tam_kb"])
print("KS entre anomalos/1024 y normales: estadistico =", round(prueba_ks.statistic, 3), " p =", round(prueba_ks.pvalue, 3))

los 70 tamanos mas grandes:
[   1269.2    1277.4    1281.5    1288.4    1296.6    1317.8    1351.6
    1396.2    1566.8   17162.2   18606.1   18821.1   40038.4   46489.6
   50288.6   62648.3   85043.2   91658.2   97361.9  121630.7  124129.3
  130252.8  130273.3  131532.8  137502.7  139786.2  141148.2  147425.3
  148377.6  149575.7  151081.   156723.2  156835.8  160983.   161085.4
  161116.2  170690.6  173649.9  175226.9  175851.5  177950.7  203192.3
  213749.8  218204.2  220303.4  226795.5  231977.   239493.1  247275.5
  258897.9  266025.   271953.9  280811.5  282593.3  289003.5  292628.5
  299182.1  315637.8  317829.1  320327.7  374681.6  376350.7  411545.6
  481976.3  498452.5  508825.6  510699.5  679936.   719155.2 1015705.6]
filas con tam_kb > 5000: 61
mayor tamano normal: 1566.78
menor tamano anomalo: 17162.24

cuantiles de los anomalos divididos por 1024:
0.00     16.8
0.25    134.3
0.50    171.7
0.75    282.2
1.00    991.9
Name: tam_kb, dtype: float64
cuantiles de los tamanos no

Los 61 tamaños gigantes, divididos por 1024, caen en el mismo rango que los tamaños normales y la prueba KS no distingue las dos distribuciones. La explicación física más simple es que esas filas se registraron en bytes en lugar de KB, así que los corregimos dividiendo por 1024 en vez de borrarlos.

Ahora revisamos la ecuación de latencia: L = red + espera + G, con espera mayor o igual a 0. Como los datos están redondeados a 0,01 ms, usamos una tolerancia de 0,02 ms.

In [23]:
tolerancia = 0.02  # ms, por el redondeo a dos decimales
residuo = crudos["latencia_ms"] - crudos["red_ms"] - crudos["gpu_ms"]  # esto deberia ser la espera en cola
latencia_valida = crudos["latencia_ms"] > 0
imposibles = latencia_valida & (residuo < -tolerancia)
print("filas con latencia < red + gpu (fuera de tolerancia):", imposibles.sum())
print("de esas, con cache_hit = 1:", (imposibles & (crudos["cache_hit"] == 1)).sum())
print("faltante de latencia en esas filas (ms):")
print(residuo[imposibles].describe().round(2))
print("filas con residuo entre -0.02 y 0 (solo redondeo):", (latencia_valida & (residuo < 0) & (residuo >= -tolerancia)).sum())

filas con latencia < red + gpu (fuera de tolerancia): 80
de esas, con cache_hit = 1: 28
faltante de latencia en esas filas (ms):
count     80.00
mean     -19.80
std       19.45
min     -134.98
25%      -21.85
50%      -14.50
75%       -9.70
max       -3.58
dtype: float64
filas con residuo entre -0.02 y 0 (solo redondeo): 4796


In [24]:
print("nulos de gpu_ms segun cache_hit:")
print(pd.crosstab(crudos["gpu_ms"].isna(), crudos["cache_hit"]))
print("nulos de red_ms segun cache_hit:")
print(pd.crosstab(crudos["red_ms"].isna(), crudos["cache_hit"]))
print("latencia = -1 segun exito:")
print(pd.crosstab(crudos["latencia_ms"] < 0, crudos["exito"]))

nulos de gpu_ms segun cache_hit:
cache_hit      0      1
gpu_ms                 
False      25951  14960
True          24     16
nulos de red_ms segun cache_hit:
cache_hit      0      1
red_ms                 
False      25946  14964
True          29     12
latencia = -1 segun exito:
exito          0      1
latencia_ms            
False        242  40587
True           0    122


Con lo encontrado aplicamos los tratamientos. La regla general es: **no borrar una solicitud que sí ocurrió**, porque las llegadas, los intentos y la caché siguen siendo información válida aunque otra columna esté mal. Solo se borran las copias (filas que no son solicitudes nuevas). Cuando un valor es imposible y no sabemos cuál de las columnas está mal, se marca como faltante (NaN) solo esa columna.

In [25]:
datos = crudos.copy()
filas_iniciales = len(datos)

datos = datos.drop_duplicates()  # 1. quitar copias exactas
quitadas_exactas = filas_iniciales - len(datos)

fila_incompleta = datos["id_solicitud"].isin(ids_raros) & datos["gpu_ms"].isna()  # 2. copia del id repetido sin gpu
datos = datos[~fila_incompleta]
quitadas_id = fila_incompleta.sum()

centinela = datos["latencia_ms"] < 0  # 3. latencia = -1 es un codigo de "no medido"
datos.loc[centinela, "latencia_ms"] = np.nan

en_bytes = datos["tam_kb"] > limite_tam  # 4. tamanos registrados en bytes
datos.loc[en_bytes, "tam_kb"] = datos.loc[en_bytes, "tam_kb"] / 1024

gpu_nula_cache = datos["gpu_ms"].isna() & (datos["cache_hit"] == 1)  # 5. con cache la gpu es 0 por definicion
datos.loc[gpu_nula_cache, "gpu_ms"] = 0.0
gpu_nula_sin_cache = datos["gpu_ms"].isna()  # 6. sin cache no se puede saber la gpu: se deja NaN
red_nula = datos["red_ms"].isna()  # 7. red faltante: se deja NaN

residuo = datos["latencia_ms"] - datos["red_ms"] - datos["gpu_ms"]
latencia_imposible = residuo < -tolerancia  # 8. latencia menor que red + gpu
datos.loc[latencia_imposible, "latencia_ms"] = np.nan

datos = datos.reset_index(drop=True)
print("filas finales:", len(datos))

filas finales: 40829


In [26]:
tabla_a1 = pd.DataFrame({
    "filas afectadas": [
        quitadas_exactas, quitadas_id, centinela.sum(), en_bytes.sum(),
        gpu_nula_cache.sum(), gpu_nula_sin_cache.sum(), red_nula.sum(), latencia_imposible.sum(), 0, 0, 0,
    ],
    "criterio": [
        "fila identica a otra en todas las columnas",
        "id repetido; una copia tiene gpu_ms nulo y la otra completa",
        "latencia_ms = -1 (codigo de 'no medido', no es un tiempo)",
        "tam_kb > 5000: hueco entre 1567 y 17162; /1024 coincide con la distribucion normal (KS)",
        "gpu_ms nulo con cache_hit = 1",
        "gpu_ms nulo con cache_hit = 0",
        "red_ms nulo",
        "latencia < red + gpu - 0.02 ms (espera en cola negativa)",
        "gpu_ms > 0 con cache_hit = 1, o gpu_ms = 0 sin cache",
        "intentos fuera de 1..3, o exito = 0 con menos de 3 intentos",
        "tiempos negativos o llegadas fuera de [0, 1200] s",
    ],
    "tratamiento": [
        "eliminar la copia",
        "conservar la fila completa",
        "latencia -> NaN; la solicitud se conserva para llegadas, intentos y gpu",
        "dividir tam_kb entre 1024 (estaba en bytes)",
        "imputar gpu_ms = 0 (por definicion del sistema)",
        "dejar NaN; se excluye solo en analisis de gpu",
        "dejar NaN; se excluye solo en analisis de red",
        "latencia -> NaN (no se sabe que columna fallo)",
        "ninguno (0 casos)",
        "ninguno (0 casos)",
        "ninguno (0 casos)",
    ],
})
print(tabla_a1.to_string(index=False))
print()
print("filas crudas:", filas_iniciales, " -> filas limpias:", len(datos))
print("latencias disponibles:", datos["latencia_ms"].notna().sum())

 filas afectadas                                                                                criterio                                                             tratamiento
             121                                              fila identica a otra en todas las columnas                                                       eliminar la copia
               1                             id repetido; una copia tiene gpu_ms nulo y la otra completa                                              conservar la fila completa
             122                               latencia_ms = -1 (codigo de 'no medido', no es un tiempo) latencia -> NaN; la solicitud se conserva para llegadas, intentos y gpu
              61 tam_kb > 5000: hueco entre 1567 y 17162; /1024 coincide con la distribucion normal (KS)                             dividir tam_kb entre 1024 (estaba en bytes)
              16                                                           gpu_ms nulo con cache_hit = 1           

**Conclusión A1.** Solo se eliminan las copias (122 filas), porque no son solicitudes nuevas e inflarían la tasa de llegadas. El resto de problemas se corrige (tamaños en bytes, GPU nula con caché) o se marca como faltante solo en la columna afectada. Las comprobaciones de coherencia entre caché y GPU, intentos y éxito, y tiempos de llegada no encontraron problemas.